In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import torch
import torch.nn as nn
import onnx
import onnx2torch
import numpy as np
from typing import Dict, Any
import sys
import os

# 添加项目路径
PROJECT_ROOT = "/inspire/hdd/global_user/hezhengfu-240208120186/rlin_projects/rlin_projects/chess-SAEs/exp/leela-interp/src"
sys.path.insert(0, str(PROJECT_ROOT))

from leela_interp import LeelaBoard


In [14]:
def compare_tensors(tensor_a: torch.Tensor, tensor_b: torch.Tensor, atol: float = 1e-2, rtol: float = 1e-2) -> bool:
    """
    比较两个张量是否接近。若形状不一致则自动view成相同形状，并都转到cuda。
    若不接近则打印第一个不接近的位置和对应的值。
    返回是否接近。
    """
    import torch
    # 转到cuda
    tensor_a = tensor_a.to('cuda')
    tensor_b = tensor_b.to('cuda')
    # 形状统一
    if tensor_a.shape != tensor_b.shape:
        try:
            tensor_b = tensor_b.view(tensor_a.shape)
        except Exception as e:
            print(f"无法view成相同形状: {tensor_a.shape} vs {tensor_b.shape}, 错误: {e}")
            return False
    # 判断是否接近
    if torch.allclose(tensor_a, tensor_b, atol=atol, rtol=rtol):
        return True
    else:
        diff = torch.abs(tensor_a - tensor_b)
        mask = diff > atol + rtol * torch.abs(tensor_b)
        if mask.any():
            idx = torch.nonzero(mask, as_tuple=True)
            first_idx = tuple(i[0].item() for i in idx)
            print("第一个不接近的位置:", first_idx)
            print("tensor_a中的值:", tensor_a[first_idx].item())
            print("tensor_b中的值:", tensor_b[first_idx].item())
            print("差值:", diff[first_idx].item())
        else:
            print("有不接近元素，但未找到具体位置")
        return False


In [15]:

class IntermediateVariableCapture:
    """捕获模型推理过程中的中间变量"""
    
    def __init__(self):
        self.intermediate_outputs: Dict[str, torch.Tensor] = {}
        self.hooks = []
    
    def hook_fn(self, name: str):
        """创建hook函数"""
        def hook(module, input, output):
            if isinstance(output, torch.Tensor):
                self.intermediate_outputs[name] = output.detach().clone()
                print(f"📍 {name}: shape={output.shape}, range=[{output.min():.6f}, {output.max():.6f}]")
            elif isinstance(output, (list, tuple)):
                for i, out in enumerate(output):
                    if isinstance(out, torch.Tensor):
                        self.intermediate_outputs[f"{name}_output_{i}"] = out.detach().clone()
                        print(f"📍 {name}_output_{i}: shape={out.shape}, range=[{out.min():.6f}, {out.max():.6f}]")
        return hook
    
    def register_hooks(self, model: nn.Module):
        """为模型的所有层注册hooks"""
        for name, module in model.named_modules():
            if name:  # 跳过根模块
                hook = module.register_forward_hook(self.hook_fn(name))
                self.hooks.append(hook)
    
    def remove_hooks(self):
        """移除所有hooks"""
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

def debug_onnx_inference_with_hooks(fen: str, onnx_model_path: str):
    """使用hooks调试ONNX推理过程"""
    
    print(f"=== 使用Hooks调试ONNX推理 ===")
    print(f"FEN: {fen}")
    
    # 1. 生成输入特征
    try:
        board = LeelaBoard.from_fen(fen, history_synthesis=True)
        features = board.lcz_features()
        print(f"✅ 特征生成成功，形状: {features.shape}")
        
        input_tensor = torch.from_numpy(features).float().unsqueeze(0)
        print(f"输入张量形状: {input_tensor.shape}")
        
    except Exception as e:
        print(f"❌ 特征生成失败: {e}")
        return
    
    # 2. 加载和转换ONNX模型
    try:
        print("\n--- 加载ONNX模型 ---")
        onnx_model = onnx.load(onnx_model_path)
        _lc0_model = onnx2torch.convert(onnx_model)
        _lc0_model.eval()
        
        print(f"模型加载成功")
        
    except Exception as e:
        print(f"❌ 模型加载失败: {e}")
        return
    
    # 3. 设置中间变量捕获
    capturer = IntermediateVariableCapture()
    capturer.register_hooks(_lc0_model)
    
    # 4. 执行推理
    try:
        print("\n--- 开始推理（带中间变量监控） ---")
        with torch.no_grad():
            outputs = _lc0_model(input_tensor)
        
        print(f"\n✅ 推理完成")
        print(f"输出数量: {len(outputs)}")
        for i, output in enumerate(outputs):
            print(f"  输出 {i}: 形状 {output.shape}, 范围 [{output.min():.6f}, {output.max():.6f}]")
        
    except Exception as e:
        print(f"❌ 推理失败: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        # 5. 清理hooks
        capturer.remove_hooks()
    
    return capturer.intermediate_outputs

def debug_specific_layers(fen: str, onnx_model_path: str, target_layers: list[str], target_params: list[str] = None):
    """只监控特定层的输出"""
    
    print(f"=== 监控特定层 ===")
    print(f"目标层: {target_layers}")
    
    # 生成输入
    board = LeelaBoard.from_fen(fen, history_synthesis=True)
    features = board.lcz_features()
    input_tensor = torch.from_numpy(features).float().unsqueeze(0)
    
    # 加载模型
    onnx_model = onnx.load(onnx_model_path)
    _lc0_model = onnx2torch.convert(onnx_model)
    _lc0_model.eval()
    
    # 只为目标层注册hooks
    intermediate_outputs = {}
    hooks = []
    
    def create_hook(name):
        def hook(module, input, output):
            if isinstance(output, torch.Tensor):
                intermediate_outputs[name] = output.detach().clone()
                print(f"🎯 {name}: shape={output.shape}")
                if name in ["attn_body/padded_input", "attn_body/reshape2"]:
                    print(f"   范围: [{output.min():.6f}, {output.max():.6f}]")
                    print(f"   前5个值: {output.flatten()[:5].tolist()}")
        return hook
    
    # 注册指定层的hooks
    for name, module in _lc0_model.named_modules():
        if any(target in name for target in target_layers):
            hook = module.register_forward_hook(create_hook(name))
            hooks.append(hook)
    
    # 打印参数信息
    param_info = {}
    if target_params:
        print("=== ONNX参数信息 ===")
        state_dict = _lc0_model.state_dict()
        for param_name in target_params:
            if param_name in state_dict:
                param = state_dict[param_name]
                param_info[param_name] = param.detach().clone()
                print(f"\n🎯 {param_name}:")
                print(f"   形状: {param.shape}, 范围: [{param.min():.6f}, {param.max():.6f}]")
                if param.numel() <= 20:
                    print(f"   数据: {param.flatten().tolist()}")
                else:
                    print(f"   前10个值: {param.flatten()[:10].tolist()}")
            else:
                print(f"❌ {param_name} 未找到")
    
    # 推理
    try:
        with torch.no_grad():
            outputs = _lc0_model(input_tensor)
        print(f"✅ 推理完成")
        print(f'{outputs= }')
    finally:
        # 清理hooks
        for hook in hooks:
            hook.remove()
    
    return intermediate_outputs, param_info

In [16]:
if __name__ == "__main__":
    fen = "2k5/4Q3/3P4/8/6p1/4p3/q1pbK3/1R6 b - - 0 32"
    onnx_model_path = "/inspire/hdd/global_user/hezhengfu-240208120186/models/chess/leela-BT4/BT4-1024x15x32h-swa-6147500.onnx"
    

    target_layers = [
        "attn_body/transpose",
        "attn_body/reshape",
        "attn_body/embedding/slice",
        "attn_body/embedding/reshape",
        "attn_body/embedding/preprocess/matmul",
        "attn_body/embedding/preprocess/add",
        "attn_body/embedding/preprocess/reshape",
        "attn_body/embedding/concat",
        "attn_body/embedding/out/reshape",
        "attn_body/matmul",
        "attn_body/add",
        "attn_body/mish",
        "attn_body/ln",
        "attn_body/ma_gating/rehape1",
        "attn_body/ma_gating/rehape2",
        "attn_body/ffn/dense1/w",
        "attn_body/ffn/dense1/b",
        "attn_body/ffn/dense2/w",
        "attn_body/ffn/dense2/b",
        "attn_body/ffn/skip",
        "attn_body_ln2",
        "encoder0/mha/Q/transpose",
        "encoder0/mha/K/transpose",
        "encoder0/mha/V/transpose",
        "encoder0/mha/QK/scale",
        "encoder0/smolgen/compress",
        "encoder0/smolgen/compress/reshape",
        "encoder0/smolgen/dense1/w",
        "encoder0/smolgen/dense1/b",
        "encoder0/smolgen/dense1/swish",
        "encoder0/smolgen/dense2/w",
        "encoder0/smolgen/dense2/b",
        "encoder0/smolgen/ln1",
        "encoder0/smolgen/ln2",
        "encoder0/smolgen/gen_from/reshape",
        "encoder0/smolgen/smol_weight_gen",
        "encoder0/smolgen_weights",
        "encoder0/mha/QKV/matmul",
        "encoder0/mha/out/reshape",
        "encoder0/mha/out/dense/w",
        "encoder0/mha/out/dense/b",
        "encoder0/alpha*input",
        "encoder0/mha/out/skip",
        "encoder0/ln1",
        "encoder0/ln2",
        "encoder0/ffn/dense1/w",
        "encoder0/ffn/dense1/b",
        "encoder0/ffn/dense2/b",
        "encoder14/ln2",
        "policy/dense1/add",
        "policy/promotion/add",
        "policy/promotion/reshape",
        "policy/promotion/concat",
        "policy/promotion/add2",
        "policy/concat",
        "policy/promotion/slice2",
        "policy/reshape",
        "output/policy",
        "output/wdl",
        "output/mlh",
        "value/embed/mish",
        "value/dense1/add",
        "value/dense2/add",
        "mlh/embed/mish",
        "mlh/dense1/mish",
        "mlh/dense2/mish",       
    ]
    
    target_params = [
        # "initializers.onnx_initializer_14",
        # "initializers.onnx_initializer_12", 
        # "initializers.onnx_initializer_13",
        # "initializers.onnx_initializer_15",
        # "initializers.onnx_initializer_16", 
        # "initializers.onnx_initializer_28",
        # "initializers.onnx_initializer_29",
        # "initializers.onnx_initializer_30",
        "initializers.onnx_initializer_0",
        "initializers.onnx_initializer_1",
        "initializers.onnx_initializer_2",
        "initializers.onnx_initializer_3",
        "initializers.onnx_initializer_4",
        "initializers.onnx_initializer_5",
        "initializers.onnx_initializer_6",
        "initializers.onnx_initializer_7",
        "initializers.onnx_initializer_8",
        "initializers.onnx_initializer_9",
        "initializers.onnx_initializer_10",
        "initializers.onnx_initializer_11",
        "initializers.onnx_initializer_1",
        "initializers.onnx_initializer_13",
        "initializers.onnx_initializer_14",
        "initializers.onnx_initializer_15",
        "initializers.onnx_initializer_16",
        "initializers.onnx_initializer_17",
        "initializers.onnx_initializer_18",
        "initializers.onnx_initializer_19",  
        "encoder0/smolgen/ln1.weight",
        # "initializers.onnx_initializer_31",
        # "initializers.onnx_initializer_32",
        "initializers.onnx_initializer_33",
        "initializers.onnx_initializer_34",
        "initializers.onnx_initializer_36",
        "initializers.onnx_initializer_37",
        "initializers.onnx_initializer_41",
        "initializers.onnx_initializer_42",
        "initializers.onnx_initializer_43",
        # "initializers.onnx_initializer_452",
    ]
    intermediate_vars, param_info = debug_specific_layers(fen, onnx_model_path, target_layers, target_params)
    

    for name, tensor in intermediate_vars.items():
        print(f"\n🔍 {name}:")
        print(f"   形状: {tensor.shape}")
        if tensor.numel() <= 20:
            print(f"   完整数据: {tensor.flatten().tolist()}")
        else:
            print(f"   前10个值: {tensor.flatten()[:10].tolist()}")
            print(f"   第140-150个值: {tensor.flatten()[140:150].tolist()}")
            print(f"   第200-210个值: {tensor.flatten()[200:210].tolist()}")

    # 分析参数信息
    print("\n" + "=" * 80)
    print("参数信息分析")
    for param_name, param_tensor in param_info.items():
        print(f"\n🔧 {param_name}:")
        print(f"   形状: {param_tensor.shape}")
        if param_tensor.numel() <= 20:
            print(f"   完整数据: {param_tensor.flatten().tolist()}")
        else:
            print(f"   前10个值: {param_tensor.flatten()[:10].tolist()}")
            print(f"   第140-150个值: {param_tensor.flatten()[140:150].tolist()}")
            print(f"   第200-210个值: {param_tensor.flatten()[200:210].tolist()}")
            
            
# print(intermediate_vars['encoder0/mha/out/reshape'])
# print(param_info["initializers.onnx_initializer_32"])

# reshape = intermediate_vars['encoder0/mha/out/reshape'].view(-1, 64, 768)
# print(f"{reshape.shape=}")
# print(f"{reshape.flatten()[50:100].tolist()=}")

# after_w_torchmatmul = torch.matmul(reshape, param_info["initializers.onnx_initializer_32"])
# print(after_w_torchmatmul)

# after_w_and = intermediate_vars['encoder0/mha/out/reshape'] @ param_info["initializers.onnx_initializer_32"]
# print(after_w_and)

# [0.0140, 0.0264, 0.0107,  ..., 0.0038, 0.0281, 0.0080],
#          [0.0141, 0.0282, 0.0113,  ..., 0.0032, 0.0274, 0.0101],
#          [0.0134, 0.0262, 0.0112,  ..., 0.0029, 0.0289, 0.0083]

# [ 0.0674,  0.0528, -0.0163,  ...,  0.0269,  0.0932,  0.0562]

    # [ 0.0087,  0.0004,  0.0282,  ..., -0.0309, -0.0100, -0.0315],
    #     [ 0.0032,  0.0123, -0.0125,  ...,  0.0155,  0.0186, -0.0058],
    #     [-0.0250, -0.0326,  0.0196,  ..., -0.0063, -0.0093,  0.0329]

=== 监控特定层 ===
目标层: ['attn_body/transpose', 'attn_body/reshape', 'attn_body/embedding/slice', 'attn_body/embedding/reshape', 'attn_body/embedding/preprocess/matmul', 'attn_body/embedding/preprocess/add', 'attn_body/embedding/preprocess/reshape', 'attn_body/embedding/concat', 'attn_body/embedding/out/reshape', 'attn_body/matmul', 'attn_body/add', 'attn_body/mish', 'attn_body/ln', 'attn_body/ma_gating/rehape1', 'attn_body/ma_gating/rehape2', 'attn_body/ffn/dense1/w', 'attn_body/ffn/dense1/b', 'attn_body/ffn/dense2/w', 'attn_body/ffn/dense2/b', 'attn_body/ffn/skip', 'attn_body_ln2', 'encoder0/mha/Q/transpose', 'encoder0/mha/K/transpose', 'encoder0/mha/V/transpose', 'encoder0/mha/QK/scale', 'encoder0/smolgen/compress', 'encoder0/smolgen/compress/reshape', 'encoder0/smolgen/dense1/w', 'encoder0/smolgen/dense1/b', 'encoder0/smolgen/dense1/swish', 'encoder0/smolgen/dense2/w', 'encoder0/smolgen/dense2/b', 'encoder0/smolgen/ln1', 'encoder0/smolgen/ln2', 'encoder0/smolgen/gen_from/reshape', 'enco

In [17]:

from transformer_lens import HookedTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig


In [18]:
# fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
# board = LeelaBoard.from_fen(fen, history_synthesis=True)
# features = board.lcz_features()
# input_tensor = torch.from_numpy(features).float().unsqueeze(0)
# print(input_tensor[0,12,0,0])

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Tuple
import onnx
import sys

# 添加项目路径
PROJECT_ROOT = "/inspire/hdd/global_user/hezhengfu-240208120186/rlin_projects/rlin_projects/chess-SAEs/exp/leela-interp/src"
sys.path.insert(0, str(PROJECT_ROOT))

from leela_interp import LeelaBoard

# 可选的依赖导入
try:
    import onnx
    import onnx2torch
    HAS_ONNX_SUPPORT = True
except ImportError:
    HAS_ONNX_SUPPORT = False


In [20]:

class Mish(nn.Module):
    """Mish激活函数"""
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

class SwiSH(nn.Module):
    """Swish激活函数"""
    def forward(self, x):
        return x * torch.sigmoid(x)

class SquaredReLU(nn.Module):
    """Squared ReLU激活函数"""
    def forward(self, x):
        return torch.square(F.relu(x))

class LayerNorm(nn.Module):
    """与nn.LayerNorm完全一致的LayerNorm实现，添加hook支持"""
    
    def __init__(self, d_model, eps=1e-3):
        super().__init__()
        self.eps = eps
        self.w = nn.Parameter(torch.ones(d_model))
        self.b = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        # 与nn.LayerNorm完全一致的实现
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        
        # 计算scale并通过hook
        scale = torch.sqrt(var + self.eps)
        
        # 标准化
        x = (x - mean) / scale
        
        # 应用权重和偏置
        return x * self.w + self.b


In [21]:

class AttentionBody(nn.Module):
    """注意力主体 - 基于ONNX实际形状信息重构"""
    
    def __init__(self, d_model: int = 1024):  # 根据实际输出改为1024
        super().__init__()
        self.d_model = d_model
        
        # embedding preprocess层 - slice取12个特征，reshape到768，然后映射到32768，再reshape到512
        # 根据 [1,64,12] -> [1,768] -> [1,32768] -> [1,64,512] 的流程
        self.embedding_preprocess = nn.Linear(768, 32768)  # 768 -> 32768
        
        # 主体线性层 - 输入624维 (112+512)，输出1024维  
        self.main_linear = nn.Linear(624, d_model)
        
        # MA gating参数 - 对应ip_mul_gate和ip_add_gate
        self.ma_gating_mul = nn.Parameter(torch.randn(64, d_model))
        self.ma_gating_add = nn.Parameter(torch.randn(64, d_model))
        
        # FFN层 - 根据实际形状 1024 -> 1536 -> 1024
        self.ffn_dense1 = nn.Linear(d_model, 1536)  # 实际是1536，不是4倍
        self.ffn_dense2 = nn.Linear(1536, d_model)
        self.ffn_alpha = nn.Parameter(torch.ones(1))
        
        # Layer normalization - 对应attn_body/ln和attn_body/ln2
        self.ln = nn.LayerNorm(d_model, eps=1e-3)
        self.ln2 = nn.LayerNorm(d_model, eps=1e-3)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch_size, 112, 8, 8] - 输入特征平面
        Returns:
            output: [batch_size, 64, d_model] - 嵌入后的特征
        """
        batch_size = x.shape[0]
        
        # Step 1: attn_body/transpose - [B, 112, 8, 8] -> [B, 8, 8, 112]
        x = x.permute(0, 2, 3, 1)
        # print(f'attn_body_transpose:{x = }')
        
        # Step 2: attn_body/reshape - [B, 8, 8, 112] -> [B, 64, 112]  
        x = x.reshape(batch_size, 64, 112)
        # print(f'attn_body/reshape:{x = }')
        
        # Step 3: attn_body/embedding/slice - 获取前12个特征 [B, 64, 12] (不是64!)
        pos_slice = x[:, :, :12]
        
        # Step 4: attn_body/embedding/reshape - [B, 64, 12] -> [B, 768] 
        pos_reshaped = pos_slice.reshape(batch_size, -1)  # [B, 768]
        
        # Step 5-6: embedding预处理 - [B, 768] -> [B, 32768] -> [B, 64, 512]
        pos_processed = self.embedding_preprocess(pos_reshaped)  # [B, 32768]
        # print(f'attn_body_embedding_preprocess:{pos_processed = }')
        pos_processed = pos_processed.reshape(batch_size, 64, 512)  # [B, 64, 512]
        # print(f'comparing attn_body/embedding/preprocess/reshape')
        # compare_tensors(pos_processed, intermediate_vars['attn_body/embedding/preprocess/reshape'])
        # Step 7: attn_body/embedding/concat - [B, 64, 112] + [B, 64, 512] -> [B, 64, 624]
        x_concat = torch.cat([x, pos_processed], dim=-1)  # [B, 64, 624]
        
        # Step 8: attn_body/embedding/out/reshape - [B, 64, 624] -> [64, 624] (去掉batch维度)
        x_concat = x_concat.reshape(-1, 624)  # [B*64, 624]
        # print(f'attn_body/embedding/out/reshape.shape:{x_concat.shape = }')
        # print(f'attn_body/embedding/out/reshape:{x_concat = }')
        # compare_tensors(x_concat, intermediate_vars['attn_body/ma_gating/rehape2'])
        
        # Step 9-10: attn_body/matmul + add - [64, 624] -> [64, 1024]
        x = self.main_linear(x_concat)  # [B*64, 1024]
        # print(f'comparing attn_body/add')
        # compare_tensors(x, intermediate_vars["attn_body/add"])
        # print(f'attn_body/matmul + add:{x.shape = }')
        # print(f'attn_body/matmul + add:{x = }')
        
        # print(f'PARAM:self.main_linear.T:{self.main_linear.weight.T.shape}')
        # print(f'PARAM:self.main_linear.T:{self.main_linear.weight.T.flatten().tolist()[:10]}')
        # print(f'PARAM:self.main_linear.T200-210:{self.main_linear.weight.T.flatten().tolist()[200:210]}')
        

        # Step 11: attn_body/mish - Mish激活
        x = F.mish(x)
        # print(f'comparing attn_body/mish')
        # compare_tensors(x, intermediate_vars['attn_body/mish'])        
        # Step 12: attn_body/ln - Layer normalization (此时还没有reshape回batch维度)
        x = self.ln(x)  # [B*64, 1024]
        # print(f'attn_body/ln:{x = }')
        # print(f'comparing attn_body/ln')
        # compare_tensors(x, intermediate_vars['attn_body/ln'])
        # Step 13: attn_body/ma_gating/rehape1 - 恢复batch维度 [64, 1024] -> [1, 64, 1024]
        x = x.reshape(batch_size, 64, self.d_model)  # [B, 64, 1024]
        # print(f'attn_body/ma_gating/rehape1:{x = }')
        # print(f'comparing attn_body/ma_gating/rehape1')
        # compare_tensors(x, intermediate_vars['attn_body/ma_gating/rehape1'])
        # Step 14-15: MA gating机制 - ip_mul_gate -> ip_add_gate
        # print(f'{self.ma_gating_mul.flatten().tolist()[:10] = }')
        x_gated = x * self.ma_gating_mul.unsqueeze(0)  # ip_mul_gate
        x_gated = x_gated + self.ma_gating_add.unsqueeze(0)  # ip_add_gate
        
        # Step 16: attn_body/ma_gating/rehape2 - [1, 64, 1024] -> [64, 1024]
        x_gated = x_gated.reshape(-1, self.d_model)  # [B*64, 1024]
        # print(f'attn_body/ma_gating/rehape2:{x_gated.flatten().tolist()[:10] = }')
        # print(f'attn_body/ma_gating/rehape2-140-150:{x_gated.flatten().tolist()[140:150] = }')
        # print(f'attn_body/ma_gating/rehape2-200-210:{x_gated.flatten().tolist()[200:210] = }')
        # print(f"{intermediate_vars['attn_body/ma_gating/rehape2'] = }")
        # print(f'comparing attn_body/ma_gating/rehape2')
        # compare_tensors(x_gated, intermediate_vars['attn_body/ma_gating/rehape2'])
        # Step 17-20: FFN部分 - dense1 -> mish -> dense2 -> alpha -> skip
        residual = x_gated  # 保存残差连接的输入 [B*64, 1024]
        
        ffn_out = self.ffn_dense1(x_gated)  # [B*64, 1536]

        ffn_out = F.mish(ffn_out)  # mish激活
        ffn_out = self.ffn_dense2(ffn_out)  # [B*64, 1024]
        # print(f'PARAM:self.ffn_dense2:{self.ffn_dense2.weight.flatten().tolist()[:10]}')
        # print(f'attn_body/ffn/dense2/b:{ffn_out.flatten().tolist()[:10] = }')
        
        # alpha缩放和跳跃连接
        x = ffn_out * self.ffn_alpha + residual  # [B*64, 1024]
        # print(f'ffn_alpha:{self.ffn_alpha = }')
        # print(f'attn_body/ffn/skip:{x.flatten().tolist()[:10] = }')
        
        
        # Step 21: attn_body/ln2 - 第二个Layer normalization
        x = self.ln2(x)  # [B*64, 1024]
        print(f'attn_body/ln2:{x = }')
        
        # 最终恢复为batch格式
        x = x.reshape(batch_size, 64, self.d_model)  # [B, 64, 1024]
        
        return x

class SmolGen(nn.Module):
    """SmolGen模块"""
    
    def __init__(self, d_model: int = 1024, n_heads: int = 32):
        super().__init__()
        self.n_heads = n_heads
        self.compress = nn.Linear(d_model, 32, bias=False)
        self.dense1 = nn.Linear(2048, 256)
        self.ln1 = nn.LayerNorm(256, eps=1e-3)
        self.dense2 = nn.Linear(256, 256 * n_heads)
        self.ln2 = nn.LayerNorm(256 * n_heads, eps=1e-3)
        self.smol_weight_gen = nn.Linear(256, 4096, bias=False)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch_size, 64, 768]
        Returns:
            weights: [batch_size, 64, 64]
        """
        batch_size, seq_len, _ = x.shape
        
        # compress
        compressed = self.compress(x)
        print(f'{compressed.shape = }')
        
        # reshape for dense1
        x_flat = compressed.view(batch_size, -1)
        print(f'{x_flat.shape = }')
        
        # dense1 + swish + ln1
        x = self.dense1(x_flat)
        # print(f'comparing encoder0/smolgen/dense1/b')
        # compare_tensors(x, intermediate_vars['encoder0/smolgen/dense1/b'])
        x = F.silu(x)
        print(f'after silu:{x.flatten().tolist()[:10] = }')
        x = self.ln1(x)
        print(f'{self.ln1.weight = }')
    
        # print(f'comparing encoder0/smolgen/ln1.weight')
        # compare_tensors(self.ln1.weight, param_info['encoder0/smolgen/ln1.weight'])
    
        print(f'after ln1:{x.flatten().tolist()[:10] = }')
        
        # dense2 + swish + ln2
        x = self.dense2(x)
        # print(f'comparing encoder0/smolgen/dense2/b')
        # compare_tensors(x, intermediate_vars['encoder0/smolgen/dense2/b'])
        x = F.silu(x)
        x = self.ln2(x)
        # print(f'after ln2:{x.flatten().tolist()[:10] = }')

        # print(f'comparing encoder0/smolgen/ln2')
        # compare_tensors(x, intermediate_vars['encoder0/smolgen/ln2'])
        
        # reshape for smol_weight_gen
        x = x.view(batch_size, self.n_heads, 256)
        # smol_weight_gen
        weights = self.smol_weight_gen(x)
        print(f'{self.smol_weight_gen.weight = }')
        print(f'{self.smol_weight_gen.weight.T = }')
        # print(f'comparing initializers.onnx_initializer_36')
        # compare_tensors(self.smol_weight_gen.weight, param_info['initializers.onnx_initializer_36'])
            
        weights = weights.view(batch_size, self.n_heads, 64, 64)
        
        return weights

class MultiHeadAttention(nn.Module):
    """多头注意力 - 集成SmolGen逻辑"""
    
    def __init__(self, d_model: int = 1024, n_heads: int = 32):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.qk_scale = nn.Parameter(torch.tensor([1.0 / (self.d_k ** 0.5)]))
        # 集成SmolGen
        self.smolgen = SmolGen(d_model, n_heads)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch_size, seq_len, d_model]
        Returns:
            attn_out: [batch_size, seq_len, d_model]
        """
        batch_size, seq_len, _ = x.shape   
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        print(f'{q.flatten().tolist()[:10] = }')
        print(f'{k.flatten().tolist()[:10] = }')
        print(f'{v.flatten().tolist()[:10] = }')
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.qk_scale

        smol_weights = self.smolgen(x)
        
        # print(f'comparing encoder0/mha/Q/transpose')
        # compare_tensors(q, intermediate_vars["encoder0/mha/Q/transpose"])   
        # print(f'comparing encoder0/mha/K/transpose')
        # compare_tensors(k, intermediate_vars["encoder0/mha/K/transpose"])   
        # print(f'comparing encoder0/mha/V/transpose')
        # compare_tensors(v, intermediate_vars["encoder0/mha/V/transpose"])   
    
        combined_scores = scores + smol_weights
        # print(f'comparing encoder0/smolgen_weights')
        # compare_tensors(combined_scores, intermediate_vars["encoder0/smolgen_weights"])   
        attn_weights = F.softmax(combined_scores, dim=-1)

        attn_out = torch.matmul(attn_weights, v)
        attn_out = attn_out

        attn_out = (
            attn_out
            .permute(0, 2, 1, 3)  # [B, S, H, head_dim]
            .contiguous()
            .view(batch_size, seq_len, self.d_model)  # [B, S, D]
        )
        
        # 输出投影
        attn_out = self.out_proj(attn_out)
  
        return attn_out

# newly added
class LC0MLP(nn.Module):
    """LC0的MLP模块，使用标准的nn.Linear层，支持直接权重加载"""
    
    def __init__(self, d_model, d_ff, activation='mish'):
        super().__init__()
        self.dense1 = nn.Linear(d_model, d_ff)
        self.dense2 = nn.Linear(d_ff, d_model)
        self.activation = activation
        
    def forward(self, x):
        x = self.dense1(x)
        
        # print(f'comparing initializers.onnx_initializer_42')
        # compare_tensors(self.dense1.weight, param_info['initializers.onnx_initializer_42'])       
        # print(f'{self.dense1.weight.T = }')
        # print(f'comparing initializers.onnx_initializer_43')
        # compare_tensors(self.dense1.bias, param_info['initializers.onnx_initializer_43'])    
        
        # print(f'comparing encoder0/ffn/dense1/b')
        # compare_tensors(x, intermediate_vars['encoder0/ffn/dense1/b'])        
        if self.activation == 'mish':
            print('mish! in mlp')
            x = F.mish(x)
        elif self.activation == 'squared_relu':
            x = torch.nn.functional.relu(x) ** 2
        else:
            raise ValueError(f"Unsupported activation: {self.activation}. Use 'mish' or 'squared_relu'")
        
        x = self.dense2(x)
        # print(f'comparing encoder0/ffn/dense2/b')
        # compare_tensors(x, intermediate_vars['encoder0/ffn/dense2/b'])    
        return x


class EncoderLayer(nn.Module):
    """编码器层"""
    
    def __init__(self, d_model: int = 1024, n_heads: int = 24, d_ff: int = 1536):
        super().__init__()
        
        self.n_heads = n_heads
        self.d_model = d_model
        self.mha = MultiHeadAttention(d_model, n_heads)

        self.ln1 = LayerNorm(d_model, eps=1e-3)
        self.ln2 = LayerNorm(d_model, eps=1e-3)
        
        # FFN layers
        self.mlp = LC0MLP(d_model, d_ff)
        
        # Alpha参数用于残差连接
        self.alpha_input = nn.Parameter(torch.ones(1))
        self.alpha_out1 = nn.Parameter(torch.ones(1))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 保存输入用于残差连接
        residual = x
        # Multi-head attention
        attn_out = self.mha(x)
        x = (attn_out * self.alpha_input) + residual
        x = self.ln1(x)
        residual2 = x
        
        
        # FFN
        ffn_out = self.mlp(x)
        # print(f'ffn_out: {ffn_out = }')
        # print(f"{ffn_out=}")
        
        # 第二个残差连接
        x = (ffn_out * self.alpha_out1) + residual2
        x = self.ln2(x) 
        # print(f'comparing encoder0/ln2')
        # compare_tensors(x, intermediate_vars['encoder0/ln2'])           
        return x

class PolicyHead(nn.Module):
    """Policy头"""
    
    def __init__(self, d_model: int = 1024, policy_dim: int = 1858):
        super().__init__()
        
        self.dense1 = nn.Linear(d_model, d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.scale = nn.Parameter(torch.ones(1))
        self.promotion = nn.Linear(d_model, 4, bias=False)
        self.indices = nn.Parameter(torch.randn(policy_dim))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch_size, 64, 768]
        Returns:
            policy_logits: [batch_size, 1858]
        """
        batch_size = x.shape[0]
        
        # Dense1层
        x = self.dense1(x)
        print(f'comparing policy/dense1/add')
        compare_tensors(x, intermediate_vars['policy/dense1/add']) 

        x = F.mish(x)

        
        # Q和K投影
        q = self.q_proj(x)
        k = self.k_proj(x)

        # 计算注意力分数
        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores * self.scale
        
        promotion_slice = k[:, 56:64, :]
        promotion_out = self.promotion(promotion_slice)

        
        promotion_out = promotion_out.transpose(1, 2)

        
        promotion_out_part1, promotion_out_part2 = torch.split(promotion_out, [3, 1], dim=1)
        
        promotion_out = torch.add(promotion_out_part1, promotion_out_part2)
        print(f'comparing policy/promotion/add')
        compare_tensors(promotion_out, intermediate_vars['policy/promotion/add']) 
        promotion_out = promotion_out.transpose(1, 2)
        
        
        promotion_out = promotion_out.reshape(x.shape[0], 1, 24)
        print(f'comparing policy/promotion/reshape')
        compare_tensors(promotion_out, intermediate_vars['policy/promotion/reshape'])        
        promotion_slice2 = scores[:, 48:56, 56:64]
        promotion_out2 = promotion_slice2.reshape(-1, 64, 1)
        promotion_out2 = torch.cat([promotion_out2, promotion_out2, promotion_out2], dim=-1)
        print(f'comparing policy/promotion/concat')
        compare_tensors(promotion_out2, intermediate_vars['policy/promotion/concat'])       
        promotion_out2 = promotion_out2.reshape(-1, 8, 24)
        
        promotion = promotion_out2 + promotion_out
        print(f'comparing policy/promotion/add2')
        compare_tensors(promotion, intermediate_vars['policy/promotion/add2'])           
        promotion = promotion.reshape(-1, 3, 64)
        
        policy = torch.cat([scores, promotion], dim=1)
        print(f'comparing policy/concat')
        compare_tensors(policy, intermediate_vars['policy/concat'])     
        policy = policy.reshape(-1, 4288)
        print(f'comparing policy/reshape')
        compare_tensors(policy, intermediate_vars['policy/reshape'])  
        indices_long = self.indices.detach().long()
        policy_logits = policy[:, indices_long]
        print(f'comparing output/policy')
        compare_tensors(policy_logits, intermediate_vars['output/policy']) 
        return policy_logits

class ValueHead(nn.Module):
    """价值头"""
    
    def __init__(self, d_model: int = 1024, d_value_head: int = 128):
        super().__init__()
        
        self.d_model = d_model
        self.embed = nn.Linear(d_model, d_value_head)
        self.dense1 = nn.Linear(d_value_head * 64, 128)
        self.dense2 = nn.Linear(128, 3)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch_size, 64, d_model]
        Returns:
            wdl: [batch_size, 3]
        """
        batch_size = x.shape[0]
        
        # 嵌入层
        x = x.view(batch_size * 64, self.d_model)
        x = self.embed(x)
        x = F.mish(x)
        print(f'comparing value/embed/mish')
        compare_tensors(x, intermediate_vars['value/embed/mish'])               
        # 重塑并通过密集层
        x = x.view(batch_size, -1)
        x = self.dense1(x)
        print(f'comparing value/dense1/add')
        compare_tensors(x, intermediate_vars['value/dense1/add'])      
        x = F.mish(x)
        
        # 输出层
        x = self.dense2(x)
        print(f'comparing value/dense2/add')
        compare_tensors(x, intermediate_vars['value/dense2/add'])
        wdl = F.softmax(x, dim=-1)
        # wdl = x
        print(f'comparing output/wdl')
        compare_tensors(wdl, intermediate_vars['output/wdl'])         
        return wdl

class MLHHead(nn.Module):
    """MLH头"""
    
    def __init__(self, d_model: int = 1024, d_mlh_head: int = 32):
        super().__init__()
        
        self.d_model = d_model
        self.embed = nn.Linear(d_model, d_mlh_head)
        self.dense1 = nn.Linear(d_mlh_head * 64, 128)
        self.dense2 = nn.Linear(128, 1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [batch_size, 64, d_model]
        Returns:
            mlh: [batch_size, 1]
        """
        batch_size = x.shape[0]
        
        # 嵌入层
        x = x.view(batch_size * 64, self.d_model)
        x = self.embed(x)
        x = F.mish(x)
        print(f'comparing mlh/embed/mish')
        compare_tensors(x, intermediate_vars['mlh/embed/mish'])        
        # 重塑并通过密集层
        x = x.view(batch_size, -1)
        x = self.dense1(x)
        x = F.mish(x)
        print(f'comparing mlh/dense1/mish')
        compare_tensors(x, intermediate_vars['mlh/dense1/mish'])
        # 输出层
        mlh = self.dense2(x)
        mlh = F.mish(mlh)
        print(f'comparing mlh/dense2/mish')
        compare_tensors(mlh, intermediate_vars['mlh/dense2/mish'])  
        print(f'comparing output/mlh')
        compare_tensors(mlh, intermediate_vars['output/mlh'])          
        return mlh

class CleanLC0Model(nn.Module):
    """
    清理后的LC0模型架构
    
    基于LC0神经网络的设计，包含：
    - 输入嵌入层（112个平面 -> 768维）
    - 15个transformer编码器层
    - 3个输出头：Policy（策略）、Value（价值）、MLH（移动左手）
    """
    
    def __init__(self, 
                 d_model: int = 1024,  # 根据ONNX实际输出修改为1024
                 n_heads: int = 32,
                 n_layers: int = 15,
                 d_ff: int = 1536,
                 max_seq_len: int = 64):
        super().__init__()
        
        # 模型配置
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.d_ff = d_ff
        self.max_seq_len = max_seq_len
        
        # 注意力主体（输入嵌入）
        self.attention_body = AttentionBody(d_model)
        
        # 编码器层
        self.encoders = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
        
        # 输出头
        self.policy_head = PolicyHead(d_model)
        self.value_head = ValueHead(d_model)
        self.mlh_head = MLHHead(d_model)
    
    def forward(self, board_features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        前向传播
        
        Args:
            board_features: [batch_size, 112, 8, 8] - 棋盘特征
            
        Returns:
            policy_logits: [batch_size, 1858] - 策略预测
            value_logits: [batch_size, 3] - 价值预测 
            mlh_logits: [batch_size, 1] - MLH预测
        """
        # 输入嵌入
        x = self.attention_body(board_features)
        
        # 编码器层
        for i, encoder in enumerate(self.encoders):
            x = encoder(x)
            if i == 14:
                print(f'layer 0 output: {x = }')
        
        # 输出头
        policy_logits = self.policy_head(x)
        value_logits = self.value_head(x)
        mlh_logits = self.mlh_head(x)
        
        return policy_logits, value_logits, mlh_logits
    
    def create_weight_mapping_from_graph(self, onnx_model: onnx.ModelProto) -> Dict[str, str]:
        """根据ONNX图结构创建权重映射"""
        mapping = {}
        
        # AttentionBody映射 - 根据forward_pass_implementation.py的实际索引
        mapping.update({
            # embedding_preprocess: attn_body/embedding/preprocess/matmul + add
            "attention_body.embedding_preprocess.weight": "initializers.onnx_initializer_4",
            "attention_body.embedding_preprocess.bias": "initializers.onnx_initializer_5",
            
            # main_linear: attn_body/matmul + add  
            "attention_body.main_linear.weight": "initializers.onnx_initializer_8",
            "attention_body.main_linear.bias": "initializers.onnx_initializer_9",
            
            # MA gating: ip_mul_gate + ip_add_gate
            "attention_body.ma_gating_mul": "initializers.onnx_initializer_11", 
            "attention_body.ma_gating_add": "initializers.onnx_initializer_12",
            
            # FFN layers: attn_body/ffn/dense1 + dense2 + alpha
            "attention_body.ffn_dense1.weight": "initializers.onnx_initializer_14",
            "attention_body.ffn_dense1.bias": "initializers.onnx_initializer_15",
            "attention_body.ffn_dense2.weight": "initializers.onnx_initializer_16", 
            "attention_body.ffn_dense2.bias": "initializers.onnx_initializer_17",
            "attention_body.ffn_alpha": "initializers.onnx_initializer_18",
            "attention_body.ln.weight": "attn_body/ln.weight",
            "attention_body.ln2.weight": "attn_body/ln2.weight",
            # Layer normalization参数将通过其他方式处理
        })
        
        # 为每个编码器层创建映射 - 第一个encoder从index 19开始
        for i in range(self.n_layers):
            encoder_prefix = f"encoders.{i}"
            base_idx = 19 + i * 28  # 根据forward_pass_implementation.py，第一个encoder从19开始
            
            # 多头注意力权重
            mapping.update({
                f"{encoder_prefix}.mha.q_proj.weight": f"initializers.onnx_initializer_{base_idx}",
                f"{encoder_prefix}.mha.q_proj.bias": f"initializers.onnx_initializer_{base_idx + 1}",
                f"{encoder_prefix}.mha.k_proj.weight": f"initializers.onnx_initializer_{base_idx + 3}",
                f"{encoder_prefix}.mha.k_proj.bias": f"initializers.onnx_initializer_{base_idx + 4}",
                f"{encoder_prefix}.mha.v_proj.weight": f"initializers.onnx_initializer_{base_idx + 6}",
                f"{encoder_prefix}.mha.v_proj.bias": f"initializers.onnx_initializer_{base_idx + 7}",
                f"{encoder_prefix}.mha.out_proj.weight": f"initializers.onnx_initializer_{base_idx + 20}",
                f"{encoder_prefix}.mha.out_proj.bias": f"initializers.onnx_initializer_{base_idx + 21}",
            })
            
            # SmolGen权重
            mapping.update({
                f"{encoder_prefix}.mha.smolgen.compress.weight": f"initializers.onnx_initializer_{base_idx + 10}",
                f"{encoder_prefix}.mha.smolgen.dense1.weight": f"initializers.onnx_initializer_{base_idx + 12}",
                f"{encoder_prefix}.mha.smolgen.dense1.bias": f"initializers.onnx_initializer_{base_idx + 13}",
                f"{encoder_prefix}.mha.smolgen.dense2.weight": f"initializers.onnx_initializer_{base_idx + 14}",
                f"{encoder_prefix}.mha.smolgen.dense2.bias": f"initializers.onnx_initializer_{base_idx + 15}",
                f"{encoder_prefix}.mha.smolgen.smol_weight_gen.weight": f"initializers.onnx_initializer_{base_idx + 17}",
            })
            
            # FFN权重
            mapping.update({
                f"{encoder_prefix}.mlp.dense1.weight": f"initializers.onnx_initializer_{base_idx + 23}",
                f"{encoder_prefix}.mlp.dense1.bias": f"initializers.onnx_initializer_{base_idx + 24}",
                f"{encoder_prefix}.mlp.dense2.weight": f"initializers.onnx_initializer_{base_idx + 25}",
                f"{encoder_prefix}.mlp.dense2.bias": f"initializers.onnx_initializer_{base_idx + 26}",
            })
            
            # Alpha参数 - 根据forward_pass_implementation.py的实际索引
            alpha_input_idx = 41 + i * 28  # encoder0:41, encoder1:69, encoder2:97...
            alpha_out1_idx = 46 + i * 28   # 估算的out1索引
            mapping.update({
                f"{encoder_prefix}.alpha_input": f"initializers.onnx_initializer_{alpha_input_idx}",
                f"{encoder_prefix}.alpha_out1": f"initializers.onnx_initializer_{alpha_out1_idx}",
            })
            
            # QK缩放参数 - 根据forward_pass_implementation.py的实际索引  
            qk_scale_idx = 28 + i * 28  # encoder0:28, encoder1:56, encoder2:84...
            mapping[f"{encoder_prefix}.mha.qk_scale"] = f"initializers.onnx_initializer_{qk_scale_idx}"
            
            # LayerNorm权重
            mapping.update({
                f"{encoder_prefix}.ln1.w": f"encoder{i}/ln1.weight",
                f"{encoder_prefix}.ln1.b": f"encoder{i}/ln1.bias",
                f"{encoder_prefix}.ln2.w": f"encoder{i}/ln2.weight",
                f"{encoder_prefix}.ln2.b": f"encoder{i}/ln2.bias",
            })
            
            # SmolGen LayerNorm权重
            mapping.update({
                f"{encoder_prefix}.mha.smolgen.ln1.weight": f"encoder{i}/smolgen/ln1.weight",
                f"{encoder_prefix}.mha.smolgen.ln1.bias": f"encoder{i}/smolgen/ln1.bias",
                f"{encoder_prefix}.mha.smolgen.ln2.weight": f"encoder{i}/smolgen/ln2.weight",
                f"{encoder_prefix}.mha.smolgen.ln2.bias": f"encoder{i}/smolgen/ln2.bias",
            })
            
            # 形状参数
            shape_indices = [2, 5, 8, 11, 14, 17, 20, 21, 23, 28, 30, 31, 42, 45, 48, 49, 51, 56, 58, 59]
            for j, shape_idx in enumerate(shape_indices):
                actual_idx = base_idx + shape_idx - 12
                if actual_idx >= 0 and actual_idx < 467:
                    mapping[f"_shape_param_{i}_{j}"] = f"initializers.onnx_initializer_{actual_idx}"
        
        # 输出头映射 - 根据forward_pass_implementation.py的实际索引
        mapping.update({
            # Policy头 - 从initializers_onnx_initializer_439开始
            "policy_head.dense1.weight": "initializers.onnx_initializer_439",
            "policy_head.dense1.bias": "initializers.onnx_initializer_440", 
            "policy_head.q_proj.weight": "initializers.onnx_initializer_441",
            "policy_head.q_proj.bias": "initializers.onnx_initializer_442",
            "policy_head.k_proj.weight": "initializers.onnx_initializer_444", 
            "policy_head.k_proj.bias": "initializers.onnx_initializer_445",
            "policy_head.scale": "initializers.onnx_initializer_447",
            "policy_head.promotion.weight": "initializers.onnx_initializer_450",
            "_policy_q_reshape_443": "initializers.onnx_initializer_443",
            "_policy_k_reshape_446": "initializers.onnx_initializer_446", 
            "_policy_promotion_slice_448": "initializers.onnx_initializer_448",
            "_policy_promotion_slice_449": "initializers.onnx_initializer_449",
            "_policy_promotion_split_451": "initializers.onnx_initializer_451",
            "_policy_promotion_reshape_452": "initializers.onnx_initializer_452",
            "_policy_promotion_slice2_453": "initializers.onnx_initializer_453",
            "_policy_promotion_slice2_454": "initializers.onnx_initializer_454",
            "_policy_promotion_reshape2_455": "initializers.onnx_initializer_455", 
            "_policy_promotion_reshape3_456": "initializers.onnx_initializer_456",
            "_policy_promotion_reshape4_457": "initializers.onnx_initializer_457",
            "_policy_reshape_458": "initializers.onnx_initializer_458",
            "policy_head.indices": "initializers.onnx_initializer_459",
            
            # Value头 - 从initializers_onnx_initializer_460开始
            "value_head.embed.weight": "initializers.onnx_initializer_460",
            "value_head.embed.bias": "initializers.onnx_initializer_461", 
            "_value_reshape_462": "initializers.onnx_initializer_462",
            "value_head.dense1.weight": "initializers.onnx_initializer_463",
            "value_head.dense1.bias": "initializers.onnx_initializer_464",
            "value_head.dense2.weight": "initializers.onnx_initializer_465",
            "value_head.dense2.bias": "initializers.onnx_initializer_466",
            
            # MLH头 - 从initializers_onnx_initializer_467开始
            "mlh_head.embed.weight": "initializers.onnx_initializer_467",
            "mlh_head.embed.bias": "initializers.onnx_initializer_468",
            "_mlh_reshape_469": "initializers.onnx_initializer_469", 
            "mlh_head.dense1.weight": "initializers.onnx_initializer_470",
            "mlh_head.dense1.bias": "initializers.onnx_initializer_471",
            "mlh_head.dense2.weight": "initializers.onnx_initializer_472",
            "mlh_head.dense2.bias": "initializers.onnx_initializer_473",
        })
        
        return mapping
    
    def load_from_onnx_model(self, onnx_model):
        """从ONNX模型加载权重"""
        onnx_state_dict = onnx_model.state_dict()
        my_state_dict = self.state_dict()
        
        # 分离initializers权重和直接模型参数
        initializers_weights = {}
        direct_weights = {}
        
        for key, tensor in onnx_state_dict.items():
            if key.startswith('initializers.'):
                initializers_weights[key] = tensor
            else:
                direct_weights[key] = tensor
        
        # 创建基于图结构的映射
        graph_mapping = self.create_weight_mapping_from_graph(onnx_model)
        
        # 执行权重加载
        matched_weights = 0
        all_onnx_weights = {**initializers_weights, **direct_weights}
        
        for name, param in self.named_parameters():
            if name in graph_mapping:
                onnx_name = graph_mapping[name]
                if onnx_name in all_onnx_weights:
                    try:
                        onnx_weight = all_onnx_weights[onnx_name]
                        if isinstance(onnx_weight, torch.Tensor):
                            # 对于QKV、out_proj、AttentionBody等权重，优先使用转置版本
                            if any(qkv in name for qkv in ['q_proj.weight', 'k_proj.weight', 'v_proj.weight', 'out_proj.weight', 'policy_head.dense1.weight', 'smol_weight_gen.weight', 'embedding_preprocess.weight']):
                            # if any(qkv in name for qkv in ['q_proj.weight', 'k_proj.weight', 'v_proj.weight']):
                                if param.shape == tuple(reversed(onnx_weight.shape)):
                                    param.data.copy_(onnx_weight.T)
                                    # print(f"✅ 转置加载: {name} <- {onnx_name}")
                                    matched_weights += 1
                                elif param.shape == onnx_weight.shape:
                                    param.data.copy_(onnx_weight)
                                    # print(f"✅ 直接加载: {name} <- {onnx_name}")
                                    matched_weights += 1
                            else:
                                # 其他权重保持原有逻辑
                                if param.shape == onnx_weight.shape:
                                    param.data.copy_(onnx_weight)
                                    matched_weights += 1
                                elif param.shape == tuple(reversed(onnx_weight.shape)):
                                    param.data.copy_(onnx_weight.T)
                                    matched_weights += 1
                    except Exception as e:
                        print(f"❌ 加载失败: {name} <- {onnx_name}: {e}")
        
        # 基于形状的自动匹配剩余权重
        used_onnx_keys = set(graph_mapping.values())
        shape_matched = 0
        
        for name, param in self.named_parameters():
            if name in graph_mapping:
                continue
            
            for onnx_name, onnx_weight in all_onnx_weights.items():
                if onnx_name in used_onnx_keys:
                    continue
                
                # 对于QKV、out_proj、AttentionBody等权重，优先尝试转置匹配
                if any(qkv in name for qkv in ['q_proj.weight', 'k_proj.weight', 'v_proj.weight', 'out_proj.weight', 'policy_head.dense1.weight', 'smol_weight_gen.weight', 'embedding_preprocess.weight']):
                    if param.shape == tuple(reversed(onnx_weight.shape)):
                        param.data.copy_(onnx_weight.T)
                        used_onnx_keys.add(onnx_name)
                        shape_matched += 1
                        print(f"🔄 自动转置匹配: {name} <- {onnx_name}")
                        break
                    elif param.shape == onnx_weight.shape:
                        param.data.copy_(onnx_weight)
                        used_onnx_keys.add(onnx_name)
                        shape_matched += 1
                        print(f"🔄 自动直接匹配: {name} <- {onnx_name}")
                        break
                else:
                    # 其他权重的原有逻辑
                    if param.shape == onnx_weight.shape:
                        param.data.copy_(onnx_weight)
                        used_onnx_keys.add(onnx_name)
                        shape_matched += 1
                        break
                    elif param.shape == tuple(reversed(onnx_weight.shape)):
                        param.data.copy_(onnx_weight.T)
                        used_onnx_keys.add(onnx_name)
                        shape_matched += 1
                        break

        total_params = len(list(self.named_parameters()))
        total_matched = matched_weights + shape_matched

        # 收集成功加载的PyTorch参数
        successfully_loaded_params = set()
        for name, param in self.named_parameters():
            if name in graph_mapping and graph_mapping[name] in all_onnx_weights:
                successfully_loaded_params.add(name)

        # 收集通过形状匹配的参数
        for name, param in self.named_parameters():
            if name not in successfully_loaded_params:
                for onnx_name, onnx_weight in all_onnx_weights.items():
                    if onnx_name not in used_onnx_keys:
                        continue
                    if param.shape == onnx_weight.shape or param.shape == tuple(reversed(onnx_weight.shape)):
                        successfully_loaded_params.add(name)
                        break

        # 未匹配的PyTorch参数
        unmatched_params = []
        for name, param in self.named_parameters():
            if name not in successfully_loaded_params:
                unmatched_params.append((name, param.shape))

        # 未使用的ONNX权重
        unused_onnx_weights = []
        for onnx_name in all_onnx_weights:
            if onnx_name not in used_onnx_keys:
                unused_onnx_weights.append((onnx_name, all_onnx_weights[onnx_name].shape))

        print(f"权重加载完成: 映射匹配 {matched_weights} 个, 形状匹配 {shape_matched} 个")
        print(f"总参数数量: {total_params}, 已匹配: {total_matched}, 未匹配: {len(unmatched_params)}")

        if unmatched_params:
            print(f"\n❌ 未匹配的PyTorch参数 ({len(unmatched_params)}个):")
            for name, shape in unmatched_params:
                print(f"  - {name}: {shape}")

        if unused_onnx_weights:
            print(f"\n⚠️ 未使用的ONNX权重 ({len(unused_onnx_weights)}个):")
            for onnx_name, shape in unused_onnx_weights[:10]:  # 只显示前10个
                print(f"  - {onnx_name}: {shape}")
            if len(unused_onnx_weights) > 10:
                print(f"  ... 还有 {len(unused_onnx_weights) - 10} 个")

        if not unmatched_params and not unused_onnx_weights:
            print("✅ 所有权重都已成功匹配!")

        return total_matched


def create_clean_model_from_onnx(onnx_model_path: str, device: str = "cuda") -> CleanLC0Model:
    """从ONNX模型创建清晰的LC0模型"""
    if not HAS_ONNX_SUPPORT:
        raise ImportError("需要 onnx 和 onnx2torch 依赖")
    
    onnx_model = onnx.load(onnx_model_path)
    converted_model = onnx2torch.convert(onnx_model)
    converted_model.to(device)
    
    # 创建清晰的模型
    clean_model = CleanLC0Model()
    clean_model.to(device)
    
    # 加载权重
    clean_model.load_from_onnx_model(converted_model)
    
    
    torch.save(clean_model.state_dict(), "/inspire/hdd/global_user/hezhengfu-240208120186/models/chess/leela-BT4/BT4.pt")
    print("权重已保存为 BT4.pt")
    
    return clean_model

if __name__ == "__main__":
    # 测试参数
    fen = "2k5/4Q3/3P4/8/6p1/4p3/q1pbK3/1R6 b - - 0 32"
    onnx_model_path = "/inspire/hdd/global_user/hezhengfu-240208120186/models/chess/leela-BT4/BT4-1024x15x32h-swa-6147500.onnx"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    try:
        # 生成输入特征
        board = LeelaBoard.from_fen(fen, history_synthesis=True)
        features = board.lcz_features()
        input_tensor = torch.from_numpy(features).float().unsqueeze(0).to(device)
        print("[0,12,0,0]:", input_tensor[0,12,0,0])
        # 创建并加载模型
        clean_model = create_clean_model_from_onnx(onnx_model_path, device)
        clean_model.eval()
        
        # 前向推理
        with torch.no_grad():
            policy_logits, value_logits, mlh_logits = clean_model(input_tensor)
        
        print(f"Policy logits shape: {policy_logits}")
        print(f"Value logits: {value_logits}")
        print(f"MLH logits: {mlh_logits}")
        
    except Exception as e:
        print(f"推理失败: {e}")
        import traceback
        traceback.print_exc()


12,0,0,: 1
[0,12,0,0]: tensor(1., device='cuda:0')
权重加载完成: 映射匹配 469 个, 形状匹配 2 个
总参数数量: 471, 已匹配: 471, 未匹配: 0

⚠️ 未使用的ONNX权重 (8个):
  - initializers.onnx_initializer_0: torch.Size([3])
  - initializers.onnx_initializer_1: torch.Size([3])
  - initializers.onnx_initializer_2: torch.Size([3])
  - initializers.onnx_initializer_3: torch.Size([2])
  - initializers.onnx_initializer_6: torch.Size([3])
  - initializers.onnx_initializer_7: torch.Size([2])
  - initializers.onnx_initializer_10: torch.Size([3])
  - initializers.onnx_initializer_13: torch.Size([2])
权重已保存为 BT4.pt
attn_body/ln2:x = tensor([[ 1.1830e-03,  1.2219e-01, -8.0799e-03,  ...,  7.7068e-02,
         -1.1515e-02, -4.7050e-02],
        [-5.5341e-03,  1.6003e-01, -1.0331e-02,  ...,  4.0256e-02,
         -1.3571e-02,  6.2966e-03],
        [ 8.4892e-05,  2.4077e-02, -9.3820e-03,  ..., -1.6341e-01,
         -2.7214e-02, -1.5782e-02],
        ...,
        [ 3.9947e-02,  2.0641e-01, -1.1262e-02,  ...,  4.3026e-03,
         -1.8423e-02, -

In [22]:
fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"